# CALCUALTE DIAGNOSIS AGE 

The script calculates both the diagnosis age for all individuals in the AoU. The duration in days is then converted to years by dividing by 365.25. 

In [1]:
# Import the packages
import os
import pandas as pd
from datetime import datetime
from google.cloud import bigquery
from functools import reduce

In [2]:
# Retrieve the value of the environment variable 'WORKSPACE_BUCKET' and assign it to the variable 'my_bucket'.
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Display the value of 'my_bucket'
my_bucket


'gs://rw-migration-aou-rw-9f879fc4'

In [18]:
time_to_event= f'{my_bucket}/data/covariates/aml_phenotype_time_to_event.tsv'
ehr_time = pd.read_csv(time_to_event, sep='\t', dtype={2: str})
ehr_time = ehr_time[~(ehr_time['concept_id'].isna())]
ehr_time = ehr_time[['person_id','entry_datetime']]

In [19]:
observation_age_period= f'{my_bucket}/data/covariates/observation_age_period.tsv'
observation = pd.read_csv(observation_age_period, sep='\t')

In [30]:
age_event = (
    ehr_time.merge(observation, 
               on ="person_id", 
               how ="inner")
)

In [31]:
age_event["date_of_birth"] = (
    pd.to_datetime(age_event["date_of_birth"]).dt.tz_localize(None)
)

In [32]:
age_event['entry_datetime'] = pd.to_datetime(age_event['entry_datetime'], errors='coerce').dt.tz_localize(None)
age_event['date_of_birth'] = pd.to_datetime(age_event['date_of_birth'], errors='coerce').dt.tz_localize(None)

age_event['age_at_dx'] = (
    (age_event['entry_datetime'] - age_event['date_of_birth']).dt.days / 365.25).fillna(age_event['age']
                                                                                       )

In [ ]:
age_event.to_csv(f"{my_bucket}/data/covariates/diagnosis_age.tsv", 
                    sep ='\t', 
                    index=False)

age_event.to_csv(f"{my_bucket}/data/covariates/diagnosis_age.tsv", 
                    sep ='\t', 
                    index=False)